# Hybrid CNN-RNN results summary

This notebook reads the CSV files generated by `01_hybrid_cnn_rnn_grid.ipynb` and creates compact tables for the report.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "util.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

DATA_OUT = PROJECT_ROOT / "data" / "hybrid"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.8f}".format)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_OUT:", DATA_OUT)

## Load generated results

In [ ]:
all_results_path = DATA_OUT / "hybrid_all_results.csv"
best_path = DATA_OUT / "hybrid_best_by_window.csv"
comparison_path = DATA_OUT / "hybrid_comparison_vs_lr.csv"

missing = [p for p in [all_results_path, best_path] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing generated files. Execute 01_hybrid_cnn_rnn_grid.ipynb first. Missing: "
        + ", ".join(str(p) for p in missing)
    )

all_results = pd.read_csv(all_results_path)
best = pd.read_csv(best_path)
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else None

print("All results shape:", all_results.shape)
print("Best by window shape:", best.shape)

## All runs

In [ ]:
display(all_results[[
    "model", "input_window", "output_window", "MAE_train", "MAE_val", "MAE_test", "params", "epochs_trained"
]].sort_values(["input_window", "output_window", "MAE_test"]))

## Best model by assigned window

In [ ]:
best_table = best[[
    "input_window", "output_window", "model", "MAE_train", "MAE_val", "MAE_test", "params", "epochs_trained"
]].sort_values(["input_window", "output_window"])

display(best_table)

best_table.to_markdown(DATA_OUT / "hybrid_best_by_window.md", index=False)
print("Markdown table saved to:", DATA_OUT / "hybrid_best_by_window.md")

## Comparison against linear regression benchmark

In [ ]:
if comparison is not None:
    comp_table = comparison[[
        "input_window", "output_window", "model", "MAE_test", "LR_MAE_test", "delta_vs_lr", "pct_delta_vs_lr", "params"
    ]].sort_values(["input_window", "output_window"])
    display(comp_table)
    comp_table.to_markdown(DATA_OUT / "hybrid_comparison_vs_lr.md", index=False)
    print("Markdown comparison saved to:", DATA_OUT / "hybrid_comparison_vs_lr.md")
else:
    print("No comparison file found.")

## Best test MAE matrix

In [ ]:
matrix = best.pivot(index="input_window", columns="output_window", values="MAE_test")
display(matrix)

matrix_path = DATA_OUT / "hybrid_test_mae_matrix.csv"
matrix.to_csv(matrix_path)
print("Matrix saved to:", matrix_path)

## Plot: best test MAE by window

In [ ]:
plot_df = best.copy()
plot_df["window"] = plot_df["input_window"].astype(str) + "-" + plot_df["output_window"].astype(str)

fig = plt.figure(figsize=(8, 4))
plt.bar(plot_df["window"], plot_df["MAE_test"])
plt.xlabel("Input-output window")
plt.ylabel("MAE test")
plt.title("Best hybrid model by assigned window")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plot_path = DATA_OUT / "hybrid_best_mae_by_window.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Plot saved to:", plot_path)